# Deep Learning for Tabular Data – Covertype Dataset

This notebook evaluates the performance of classical and deep learning models on the Covertype dataset.

Models:
- LightGBM (baseline)
- TabNet
- FT-Transformer

Evaluation metrics:
- Accuracy
- Macro-F1 score

Data split:
- 60% train / 20% validation / 20% test

## 1. Data Loading

Load the Covertype dataset and inspect its basic structure.

In [2]:
import pandas as pd

df = pd.read_csv("C:/Users/Jerry/Desktop/covertype/covtype.data", header=None)
df.shape

(581012, 55)

## 2. Data Preparation

Separate features (X) and labels (y).  
The target variable is adjusted to start from 0 for compatibility with PyTorch models.

In [3]:
X = df.iloc[:, :-1]
y = df.iloc[:, -1] - 1

In [4]:
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Classes:", sorted(y.unique()))
print("\nClass distribution:")
print(y.value_counts().sort_index())

print("\nMissing values in X:")
print(X.isnull().sum().sum())

X shape: (581012, 54)
y shape: (581012,)
Classes: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]

Class distribution:
54
0    211840
1    283301
2     35754
3      2747
4      9493
5     17367
6     20510
Name: count, dtype: int64

Missing values in X:
0


## 3. Train / Validation / Test Split

Split the dataset into 60% training, 20% validation, and 20% test sets using a fixed random seed for reproducibility.

In [5]:
from sklearn.model_selection import train_test_split

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.25,
    random_state=42,
    stratify=y_temp
)

print("Train:", X_train.shape, y_train.shape)
print("Val:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Train: (348606, 54) (348606,)
Val: (116203, 54) (116203,)
Test: (116203, 54) (116203,)


In [6]:
from sklearn.preprocessing import StandardScaler

X_train_raw = X_train.copy()
X_val_raw = X_val.copy()
X_test_raw = X_test.copy()

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Raw train shape:", X_train_raw.shape)
print("Scaled train shape:", X_train_scaled.shape)

Raw train shape: (348606, 54)
Scaled train shape: (348606, 54)


## 4. Evaluation Setup

Define evaluation metrics (accuracy and macro-F1) and initialize a results container for model comparison.

In [7]:
from sklearn.metrics import accuracy_score, f1_score

def evaluate_classification(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro")
    }

results = []

## 5. LightGBM Baseline

Train a tree-based gradient boosting model as a classical baseline.  
This model typically performs strongly on tabular data with minimal preprocessing.

In [8]:
import lightgbm as lgb
import time

from sklearn.metrics import accuracy_score, f1_score

start_time = time.time()

lgb_model = lgb.LGBMClassifier(
    objective="multiclass",
    num_class=7,
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=64,
    random_state=42,
    n_jobs=-1
)

lgb_model.fit(
    X_train_raw, y_train,
    eval_set=[(X_val_raw, y_val)],
    eval_metric="multi_logloss"
)

train_time = time.time() - start_time

start_inf = time.time()
y_val_pred = lgb_model.predict(X_val_raw)
inference_time = time.time() - start_inf

metrics = evaluate_classification(y_val, y_val_pred)

results.append({
    "model": "LightGBM",
    "accuracy": metrics["accuracy"],
    "macro_f1": metrics["macro_f1"],
    "train_time": train_time,
    "inference_time": inference_time
})

print("LightGBM validation results:")
print(metrics)
print(f"Training time: {train_time:.2f} sec")
print(f"Inference time: {inference_time:.4f} sec")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.019677 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2268
[LightGBM] [Info] Number of data points in the train set: 348606, number of used features: 53
[LightGBM] [Info] Start training from score -1.008937
[LightGBM] [Info] Start training from score -0.718262
[LightGBM] [Info] Start training from score -2.788124
[LightGBM] [Info] Start training from score -5.353773
[LightGBM] [Info] Start training from score -4.114354
[LightGBM] [Info] Start training from score -3.510215
[LightGBM] [Info] Start training from score -3.343855
LightGBM validation results:
{'accuracy': 0.9310602996480297, 'macro_f1': 0.9222527684832158}
Training time: 51.35 sec
Inference time: 17.0093 sec


## 6. Logistic Regression Baseline

As a simple linear model, Logistic Regression is used as a lightweight baseline to compare against tree-based and deep learning models. 

Unlike LightGBM, Logistic Regression assumes linear relationships between features and the target, making it less flexible but more interpretable. It also requires feature scaling for optimal performance.

This model helps evaluate whether complex models are truly necessary for this dataset.

In [11]:
from sklearn.linear_model import LogisticRegression
import time

start_time = time.time()

lr_model = LogisticRegression(max_iter=1000, n_jobs=-1)

lr_model.fit(X_train, y_train)

train_time = time.time() - start_time

start_inf = time.time()
y_val_pred = lr_model.predict(X_val)
inference_time = time.time() - start_inf

metrics = evaluate_classification(y_val, y_val_pred)

results.append({
    "model": "Logistic Regression",
    "accuracy": metrics["accuracy"],
    "macro_f1": metrics["macro_f1"],
    "train_time": train_time,
    "inference_time": inference_time
})

print("Logistic Regression validation results:")
print(metrics)
print(f"Training time: {train_time:.2f} sec")
print(f"Inference time: {inference_time:.4f} sec")

c:\Users\Jerry\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Logistic Regression validation results:
{'accuracy': 0.6858773009302686, 'macro_f1': 0.35907115303898507}
Training time: 152.70 sec
Inference time: 0.0860 sec


c:\Users\Jerry\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


### 7. Hyperparameter Tuning with Optuna

Use Optuna to search for better hyperparameters and improve model performance.LightGBM performance is sensitive to hyperparameter choices. Therefore, Optuna is used to automatically search for optimal parameters.

In [12]:
import optuna

def objective_lgb(trial):
    params = {
        "objective": "multiclass",
        "num_class": 7,
        "metric": "multi_logloss",
        "boosting_type": "gbdt",
        "n_estimators": trial.suggest_int("n_estimators", 200, 1000),
        "learning_rate": trial.suggest_float("learning_rate", 1e-2, 0.2, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 31, 255),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "random_state": 42,
        "n_jobs": -1,
        "verbosity": -1
    }

    model = lgb.LGBMClassifier(**params)
    model.fit(X_train_raw, y_train)

    y_val_pred = model.predict(X_val_raw)
    macro_f1 = f1_score(y_val, y_val_pred, average="macro")

    return macro_f1

c:\Users\Jerry\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 8. TabNet Model

Train a deep learning model specifically designed for tabular data.

TabNet uses feature selection through attention mechanisms and is designed to handle tabular data efficiently.

In [13]:
study_lgb = optuna.create_study(direction="maximize")
study_lgb.optimize(objective_lgb, n_trials=20)

print("Best trial:")
print("Best macro-F1:", study_lgb.best_value)
print("Best params:")
print(study_lgb.best_params)

[I 2026-04-06 00:25:49,723] A new study created in memory with name: no-name-afda6c50-e4e9-432f-b81d-d93522fa53c9
[I 2026-04-06 00:26:33,065] Trial 0 finished with value: 0.9229690907195328 and parameters: {'n_estimators': 299, 'learning_rate': 0.06570212152163757, 'num_leaves': 153, 'max_depth': 11, 'min_child_samples': 13, 'subsample': 0.6995166563071528, 'colsample_bytree': 0.9057507903983022, 'reg_alpha': 0.20566574047759645, 'reg_lambda': 0.013146497818088583}. Best is trial 0 with value: 0.9229690907195328.
[I 2026-04-06 00:26:58,677] Trial 1 finished with value: 0.8854932630495266 and parameters: {'n_estimators': 213, 'learning_rate': 0.04541172712967566, 'num_leaves': 142, 'max_depth': 10, 'min_child_samples': 73, 'subsample': 0.9050397883767661, 'colsample_bytree': 0.732269262850154, 'reg_alpha': 0.001744959499872421, 'reg_lambda': 2.2909613261697145e-05}. Best is trial 0 with value: 0.9229690907195328.
[I 2026-04-06 00:29:05,090] Trial 2 finished with value: 0.927834145515676

Best trial:
Best macro-F1: 0.9395557548022008
Best params:
{'n_estimators': 791, 'learning_rate': 0.1042774141584439, 'num_leaves': 100, 'max_depth': 11, 'min_child_samples': 81, 'subsample': 0.7611080035536162, 'colsample_bytree': 0.8433060707881544, 'reg_alpha': 0.00010055422908363006, 'reg_lambda': 0.0007162221839817648}


In [14]:
import torch
from pytorch_tabnet.tab_model import TabNetClassifier

train_X_tab = X_train.to_numpy()
val_X_tab = X_val.to_numpy()

train_y_tab = y_train.to_numpy()
val_y_tab = y_val.to_numpy()

start_time = time.time()

tabnet_model = TabNetClassifier(
    n_d=16,
    n_a=16,
    n_steps=5,
    gamma=1.5,
    lambda_sparse=1e-4,
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=2e-2),
    mask_type="sparsemax"
)

tabnet_model.fit(
    train_X_tab, train_y_tab,
    eval_set=[(val_X_tab, val_y_tab)],
    eval_name=["val"],
    eval_metric=["accuracy"],
    max_epochs=30,
    patience=10,
    batch_size=4096,
    virtual_batch_size=512
)

train_time = time.time() - start_time

start_inf = time.time()
val_pred = tabnet_model.predict(val_X_tab)
inference_time = time.time() - start_inf

metrics = evaluate_classification(val_y_tab, val_pred)

results.append({
    "model": "TabNet",
    "accuracy": metrics["accuracy"],
    "macro_f1": metrics["macro_f1"],
    "train_time": train_time,
    "inference_time": inference_time
})

print("TabNet validation results:")
print(metrics)
print(f"Training time: {train_time:.2f} sec")
print(f"Inference time: {inference_time:.4f} sec")
print(f"Average time per sample: {inference_time / len(val_X_tab):.8f} sec")

c:\Users\Jerry\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pytorch_tabnet\abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 1.04848 | val_accuracy: 0.50574 |  0:00:28s
epoch 1  | loss: 0.75683 | val_accuracy: 0.44942 |  0:00:56s
epoch 2  | loss: 0.64718 | val_accuracy: 0.64433 |  0:01:24s
epoch 3  | loss: 0.60671 | val_accuracy: 0.71981 |  0:01:53s
epoch 4  | loss: 0.57872 | val_accuracy: 0.7386  |  0:02:22s
epoch 5  | loss: 0.56398 | val_accuracy: 0.76378 |  0:02:52s
epoch 6  | loss: 0.54639 | val_accuracy: 0.77476 |  0:03:24s
epoch 7  | loss: 0.53906 | val_accuracy: 0.77387 |  0:03:55s
epoch 8  | loss: 0.52097 | val_accuracy: 0.79233 |  0:04:27s
epoch 9  | loss: 0.50691 | val_accuracy: 0.79524 |  0:04:59s
epoch 10 | loss: 0.49088 | val_accuracy: 0.78897 |  0:05:26s
epoch 11 | loss: 0.48374 | val_accuracy: 0.80142 |  0:05:55s
epoch 12 | loss: 0.46886 | val_accuracy: 0.80907 |  0:06:28s
epoch 13 | loss: 0.45981 | val_accuracy: 0.81771 |  0:07:03s
epoch 14 | loss: 0.46298 | val_accuracy: 0.81741 |  0:07:38s
epoch 15 | loss: 0.44329 | val_accuracy: 0.82415 |  0:08:12s
epoch 16 | loss: 0.44409

c:\Users\Jerry\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


TabNet validation results:
{'accuracy': 0.858359939072141, 'macro_f1': 0.6949273466316825}
Training time: 979.68 sec
Inference time: 6.6201 sec
Average time per sample: 0.00005697 sec


## 9. Feature Scaling for Deep Models

Apply standardization to numerical features for FT-Transformer training.

In [15]:
scaler = StandardScaler()

train_X_scaled = scaler.fit_transform(X_train)
val_X_scaled = scaler.transform(X_val)
test_X_scaled = scaler.transform(X_test)

print(train_X_scaled.shape, val_X_scaled.shape, test_X_scaled.shape)

(348606, 54) (116203, 54) (116203, 54)


## 10. Feature Scaling and Tensor Preparation

Prepare input features for FT-Transformer by applying standardization and converting data into PyTorch tensors.

In [16]:
train_X_scaled = scaler.fit_transform(X_train)
val_X_scaled = scaler.transform(X_val)
test_X_scaled = scaler.transform(X_test)

train_X_ft = torch.tensor(train_X_scaled, dtype=torch.float32)
val_X_ft = torch.tensor(val_X_scaled, dtype=torch.float32)
test_X_ft = torch.tensor(test_X_scaled, dtype=torch.float32)

train_y_ft = torch.tensor(y_train.to_numpy(), dtype=torch.long)
val_y_ft = torch.tensor(y_val.to_numpy(), dtype=torch.long)
test_y_ft = torch.tensor(y_test.to_numpy(), dtype=torch.long)

print(train_X_ft.shape, val_X_ft.shape, test_X_ft.shape)

torch.Size([348606, 54]) torch.Size([116203, 54]) torch.Size([116203, 54])


## 11. FT-Transformer Model

Define and train a Transformer-based model adapted for tabular data.

Due to computational constraints, training is performed on a subset of the training data.

In [17]:
class NumericalFeatureTokenizer(torch.nn.Module):
    def __init__(self, n_features, d_token):
        super().__init__()
        self.weight = torch.nn.Parameter(torch.randn(n_features, d_token))
        self.bias = torch.nn.Parameter(torch.randn(n_features, d_token))

    def forward(self, x):
        return x.unsqueeze(-1) * self.weight.unsqueeze(0) + self.bias.unsqueeze(0)


class FTTransformer(torch.nn.Module):
    def __init__(self, n_features, n_classes, d_token=32, n_heads=8, n_layers=3, dropout=0.1):
        super().__init__()
        self.tokenizer = NumericalFeatureTokenizer(n_features, d_token)
        self.cls_token = torch.nn.Parameter(torch.randn(1, 1, d_token))

        encoder_layer = torch.nn.TransformerEncoderLayer(
            d_model=d_token,
            nhead=n_heads,
            dim_feedforward=d_token * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )
        self.encoder = torch.nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        self.head = torch.nn.Sequential(
            torch.nn.LayerNorm(d_token),
            torch.nn.ReLU(),
            torch.nn.Linear(d_token, n_classes)
        )

    def forward(self, x):
        x = self.tokenizer(x)
        cls = self.cls_token.expand(x.size(0), -1, -1)
        x = torch.cat([cls, x], dim=1)
        x = self.encoder(x)
        cls_out = x[:, 0]
        return self.head(cls_out)

In [18]:
train_dataset = torch.utils.data.TensorDataset(train_X_ft, train_y_ft)
val_dataset = torch.utils.data.TensorDataset(val_X_ft, val_y_ft)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=1024, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=2048, shuffle=False)

In [19]:
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

subset_size = 80000
train_X_ft_sub = train_X_ft[:subset_size]
train_y_ft_sub = train_y_ft[:subset_size]

train_dataset = torch.utils.data.TensorDataset(train_X_ft_sub, train_y_ft_sub)
val_dataset = torch.utils.data.TensorDataset(val_X_ft, val_y_ft)

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=2048,
    shuffle=True
)

val_loader = torch.utils.data.DataLoader(
    val_dataset,
    batch_size=4096,
    shuffle=False
)

ft_model = FTTransformer(
    n_features=train_X_ft_sub.shape[1],
    n_classes=7,
    d_token=16,
    n_heads=4,
    n_layers=2,
    dropout=0.1
).to(device)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(ft_model.parameters(), lr=1e-3, weight_decay=1e-5)

n_epochs = 8
best_val_f1 = -1
best_state = None

start_time = time.time()

for epoch in range(n_epochs):
    ft_model.train()
    train_loss = 0.0

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        logits = ft_model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    ft_model.eval()
    val_preds = []
    val_true = []

    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device)
            logits = ft_model(xb)
            preds = torch.argmax(logits, dim=1).cpu().numpy()

            val_preds.extend(preds)
            val_true.extend(yb.numpy())

    val_metrics = evaluate_classification(np.array(val_true), np.array(val_preds))

    print(
        f"Epoch {epoch+1}/{n_epochs} | "
        f"Train Loss: {train_loss/len(train_loader):.4f} | "
        f"Val Acc: {val_metrics['accuracy']:.4f} | "
        f"Val Macro-F1: {val_metrics['macro_f1']:.4f}"
    )

    if val_metrics["macro_f1"] > best_val_f1:
        best_val_f1 = val_metrics["macro_f1"]
        best_state = ft_model.state_dict()

train_time = time.time() - start_time
print(f"FT-Transformer training finished in {train_time:.2f} sec")

Using device: cpu


C:\Users\Jerry\AppData\Local\Temp\ipykernel_16012\4269827045.py:26: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = torch.nn.TransformerEncoder(encoder_layer, num_layers=n_layers)


Epoch 1/8 | Train Loss: 1.5245 | Val Acc: 0.4876 | Val Macro-F1: 0.0936
Epoch 2/8 | Train Loss: 1.2875 | Val Acc: 0.4876 | Val Macro-F1: 0.0936
Epoch 3/8 | Train Loss: 1.2305 | Val Acc: 0.4876 | Val Macro-F1: 0.0936
Epoch 4/8 | Train Loss: 1.2104 | Val Acc: 0.4876 | Val Macro-F1: 0.0936
Epoch 5/8 | Train Loss: 1.1410 | Val Acc: 0.6293 | Val Macro-F1: 0.1943
Epoch 6/8 | Train Loss: 0.9528 | Val Acc: 0.6837 | Val Macro-F1: 0.2933
Epoch 7/8 | Train Loss: 0.8497 | Val Acc: 0.6902 | Val Macro-F1: 0.2988
Epoch 8/8 | Train Loss: 0.7989 | Val Acc: 0.6954 | Val Macro-F1: 0.3015
FT-Transformer training finished in 989.24 sec


## 12. Evaluation and Results

Evaluate the FT-Transformer model on the validation set and record performance metrics, training time, and inference time.
Tree-based models such as LightGBM are more interpretable due to feature importance,while deep learning models sacrifice interpretability for flexibility.
LightGBM performance is sensitive to hyperparameter choices. Therefore, Optuna is used to automatically search for optimal parameters.

In [20]:
ft_model.load_state_dict(best_state)
ft_model.eval()

start_inf = time.time()

val_preds = []
with torch.no_grad():
    for xb, _ in val_loader:
        xb = xb.to(device)
        logits = ft_model(xb)
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        val_preds.extend(preds)

inference_time = time.time() - start_inf

metrics = evaluate_classification(y_val.to_numpy(), np.array(val_preds))

results.append({
    "model": "FT-Transformer-subset",
    "accuracy": metrics["accuracy"],
    "macro_f1": metrics["macro_f1"],
    "train_time": train_time,
    "inference_time": inference_time
})

print("FT-Transformer validation results:")
print(metrics)
print(f"Training time: {train_time:.2f} sec")
print(f"Inference time: {inference_time:.4f} sec")
print(f"Average time per sample: {inference_time / len(val_X_ft):.8f} sec")

FT-Transformer validation results:
{'accuracy': 0.6953607049731935, 'macro_f1': 0.30153862936370796}
Training time: 989.24 sec
Inference time: 20.8884 sec
Average time per sample: 0.00017976 sec


In [21]:
pd.DataFrame(results)

,model,accuracy,macro_f1,train_time,inference_time
0,LightGBM,0.931060,0.922253,51.352001,17.009308
1,Logistic Regression,0.685877,0.359071,152.701716,0.085954
2,TabNet,0.858360,0.694927,979.677704,6.620137
3,FT-Transformer-subset,0.695361,0.301539,989.240324,20.888371
